# GPA Predictor — Existential Factors vs. Academic Performance

This notebook analyzes the [Students Performance Dataset](https://www.kaggle.com/datasets/rabieelkharoua/students-performance-dataset/data) (2,392 high school students) to identify which factors most strongly predict GPA, and compares several regression models to find the best predictor.

Full write-up of methodology and findings: see `FINAL_Research_Paper.pdf` in this repo.

**Setup:** download `Student_performance_data _.csv` from the Kaggle link above and place it in this notebook's directory before running.

## 1. Load and Explore the Data

In [ ]:
import pandas as pd

data = pd.read_csv('Student_performance_data _.csv')
data.head()

In [ ]:
# Check for missing values
data.isnull().sum()

## 2. Explore Candidate Factors

Before modeling, we visually check which independent variables show any relationship with GPA. Factors with no visible correlation are dropped from the final feature set.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Factors with no clear relationship to GPA — kept for comparison, later dropped
for col in ['Extracurricular', 'Sports', 'Volunteering', 'Gender']:
    sns.scatterplot(x=data[col], y=data['GPA'])
    plt.title(f'{col} vs. GPA')
    plt.show()

In [ ]:
# Absences shows a clear negative relationship with GPA
sns.scatterplot(x=data['Absences'], y=data['GPA'])
plt.title('Absences vs. GPA')
plt.show()

**Observation:** Extracurriculars, sports, volunteering, and gender show no visible correlation with GPA — students in both groups span the full 0–4 GPA range. Absences, on the other hand, shows a clear downward trend: more absences consistently track with lower GPA.

Based on this exploration (and further testing not shown here), the final feature set was narrowed to: **study time (weekly), parental support, absences, tutoring, and parental education.**

## 3. Prepare Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

features = ['StudyTimeWeekly', 'ParentalSupport', 'Absences', 'Tutoring', 'ParentalEducation']
x = data[features]
y = data['GPA']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.shape, x_test.shape

## 4. Compare Regression Models

We evaluate several regression approaches using Mean Absolute Error (MAE), Mean Squared Error (MSE), and R² (proportion of variance in GPA explained by the model).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(name, model, x_train, y_train, x_test, y_test):
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f'{name:>20} | MAE: {mae:.3f} | MSE: {mse:.3f} | R²: {r2:.3f}')
    return {'model': name, 'mae': mae, 'mse': mse, 'r2': r2}

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

results = []
results.append(evaluate('Linear Regression', LinearRegression(), x_train, y_train, x_test, y_test))
results.append(evaluate('Ridge (alpha=1.0)', Ridge(alpha=1.0), x_train, y_train, x_test, y_test))
results.append(evaluate('Lasso (alpha=0.0)', Lasso(alpha=0.0), x_train, y_train, x_test, y_test))
results.append(evaluate('Random Forest', RandomForestRegressor(max_depth=2, n_estimators=100, random_state=0), x_train, y_train, x_test, y_test))
results.append(evaluate('Decision Tree', DecisionTreeRegressor(random_state=0), x_train, y_train, x_test, y_test))
results.append(evaluate('SVR', make_pipeline(StandardScaler(), SVR(C=1.0, epsilon=0.2)), x_train, y_train, x_test, y_test))

In [ ]:
results_df = pd.DataFrame(results).sort_values('r2', ascending=False).reset_index(drop=True)
results_df

## 5. Results

**Ridge Regression performed best**, with the highest R² and lowest error across all models tested — meaning it explained the largest share of variance in GPA using our five selected features.

Full discussion of these results, hyperparameter tuning (alpha values for Ridge/Lasso, `n_estimators` for Random Forest), limitations, and related work is available in `FINAL_Research_Paper.pdf`.

**Key takeaway:** Study time, parental support, absences, tutoring, and parental education together are strong predictors of student GPA — while extracurriculars, sports, volunteering, and gender show no meaningful relationship.